Import libraries and configure project paths

In [1]:
from pathlib import Path
import pandas as pd

# Locate project directory
PROJECT_ROOT = Path.cwd()

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "paddy_production.csv"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

print("Project directory:", PROJECT_ROOT)
print("Raw dataset:", RAW_PATH)
print("Processed directory:", PROCESSED_DIR)

Project directory: D:\UOM\L4S1\CM4371 - Machine Learning & Pattern Recognition\MLPR_Project
Raw dataset: D:\UOM\L4S1\CM4371 - Machine Learning & Pattern Recognition\MLPR_Project\data\raw\paddy_production.csv
Processed directory: D:\UOM\L4S1\CM4371 - Machine Learning & Pattern Recognition\MLPR_Project\data\processed


Configuration

In [2]:
# Target variable
TARGET = "Production 000 Mt."

# Lag values
LAGS = [1, 2, 3]

# Training/testing split
TEST_SIZE = 0.20

print("Target variable:", TARGET)
print("Lag features:", LAGS)
print("Test size:", TEST_SIZE)

Target variable: Production 000 Mt.
Lag features: [1, 2, 3]
Test size: 0.2


Load and validate data

In [3]:
def load_and_validate_data():

    df = pd.read_csv(RAW_PATH)

    df.columns = df.columns.str.strip()

    required_columns = [
        "Year",
        TARGET
    ]

    for column in required_columns:

        if column not in df.columns:

            raise ValueError(
                f"Missing column: {column}"
            )

    df = df[required_columns].copy()

    # Convert values to numeric
    df["Year"] = pd.to_numeric(
        df["Year"],
        errors="coerce"
    )

    df[TARGET] = pd.to_numeric(
        df[TARGET],
        errors="coerce"
    )

    # Remove invalid records
    df = df.dropna(
        subset=["Year", TARGET]
    )

    # Convert year into integer
    df["Year"] = df["Year"].astype(int)

    # Sort chronologically
    df = df.sort_values("Year")

    # Check duplicate years
    if df["Year"].duplicated().any():

        raise ValueError(
            "Duplicate years detected!"
        )

    # Check consecutive years
    expected_years = list(
        range(
            df["Year"].min(),
            df["Year"].max() + 1
        )
    )

    actual_years = df["Year"].tolist()

    if actual_years != expected_years:

        raise ValueError(
            "Missing or unordered years detected!"
        )

    # Production must be positive
    if (df[TARGET] <= 0).any():

        raise ValueError(
            "Invalid production values detected!"
        )

    return df

Execute and validation

In [4]:
df = load_and_validate_data()

print("Dataset validation successful!")

print("\nNumber of records:", len(df))

print(
    "Historical period:",
    df["Year"].min(),
    "-",
    df["Year"].max()
)

display(df.head())

Dataset validation successful!

Number of records: 61
Historical period: 1952 - 2012


,Year,Production 000 Mt.
0,1952,604
1,1953,458
2,1954,650
3,1955,747
4,1956,574


Create lag features

In [5]:
def create_features(df):

    data = df.copy()

    # Previous production values
    for lag in LAGS:

        data[f"production_lag_{lag}"] = (
            data[TARGET].shift(lag)
        )

    # Previous 3-year average
    data["rolling_mean_3"] = (
        data[TARGET]
        .shift(1)
        .rolling(window=3)
        .mean()
    )

    # Previous 3-year standard deviation
    data["rolling_std_3"] = (
        data[TARGET]
        .shift(1)
        .rolling(window=3)
        .std()
    )

    return data.dropna().reset_index(drop=True)

Execute feature engineering

In [6]:
featured_data = create_features(df)

print("Feature engineering completed!")

print("Original records:", len(df))

print(
    "Records after feature engineering:",
    len(featured_data)
)

print("\nGenerated columns:")
print(featured_data.columns.tolist())

display(featured_data.head())

Feature engineering completed!
Original records: 61
Records after feature engineering: 58

Generated columns:
['Year', 'Production 000 Mt.', 'production_lag_1', 'production_lag_2', 'production_lag_3', 'rolling_mean_3', 'rolling_std_3']


,Year,Production 000 Mt.,production_lag_1,production_lag_2,production_lag_3,rolling_mean_3,rolling_std_3
0,1955,747,650.0,458.0,604.0,570.666667,100.246363
1,1956,574,747.0,650.0,458.0,618.333333,147.079344
2,1957,654,574.0,747.0,650.0,657.000000,86.712168
3,1958,765,654.0,574.0,747.0,658.333333,86.581368
4,1959,761,765.0,654.0,574.0,664.333333,95.918368


Chronological train/ test split

In [7]:
def split_data(data):

    split_index = int(
        len(data) * (1 - TEST_SIZE)
    )

    train_data = data.iloc[:split_index].copy()

    test_data = data.iloc[split_index:].copy()

    return train_data, test_data

Execute train/ test split

In [8]:
train_data, test_data = split_data(featured_data)

print("Training records:", len(train_data))
print("Testing records:", len(test_data))

print(
    "\nTraining period:",
    train_data["Year"].min(),
    "-",
    train_data["Year"].max()
)

print(
    "Testing period:",
    test_data["Year"].min(),
    "-",
    test_data["Year"].max()
)

Training records: 46
Testing records: 12

Training period: 1955 - 2000
Testing period: 2001 - 2012


Save processed datasets

In [9]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save cleaned historical data
df.to_csv(
    PROCESSED_DIR / "paddy_cleaned.csv",
    index=False
)

# Save training dataset
train_data.to_csv(
    PROCESSED_DIR / "train_data.csv",
    index=False
)

# Save testing dataset
test_data.to_csv(
    PROCESSED_DIR / "test_data.csv",
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


Display preprocessing result

In [10]:
print("========== PREPROCESSING RESULTS ==========")

print("Original records:", len(df))

print(
    "Records after feature engineering:",
    len(featured_data)
)

print("Training records:", len(train_data))
print("Testing records:", len(test_data))

print(
    "\nTraining period:",
    train_data["Year"].min(),
    "-",
    train_data["Year"].max()
)

print(
    "Testing period:",
    test_data["Year"].min(),
    "-",
    test_data["Year"].max()
)

print("\nTraining sample:")
display(train_data.head())

print("\nTesting sample:")
display(test_data.head())

print("\nDataset preparation completed!")

========== PREPROCESSING RESULTS ==========
Original records: 61
Records after feature engineering: 58
Training records: 46
Testing records: 12

Training period: 1955 - 2000
Testing period: 2001 - 2012

Training sample:


,Year,Production 000 Mt.,production_lag_1,production_lag_2,production_lag_3,rolling_mean_3,rolling_std_3
0,1955,747,650.0,458.0,604.0,570.666667,100.246363
1,1956,574,747.0,650.0,458.0,618.333333,147.079344
2,1957,654,574.0,747.0,650.0,657.000000,86.712168
3,1958,765,654.0,574.0,747.0,658.333333,86.581368
4,1959,761,765.0,654.0,574.0,664.333333,95.918368



Testing sample:


,Year,Production 000 Mt.,production_lag_1,production_lag_2,production_lag_3,rolling_mean_3,rolling_std_3
46,2001,2695,2860.0,2857.0,2692.0,2803.000000,96.140522
47,2002,2860,2695.0,2860.0,2857.0,2804.000000,94.408686
48,2003,3067,2860.0,2695.0,2860.0,2805.000000,95.262794
49,2004,2628,3067.0,2860.0,2695.0,2874.000000,186.394742
50,2005,3246,2628.0,3067.0,2860.0,2851.666667,219.618609



Dataset preparation completed!
